In [ ]:
# ================= INSTALL DEPENDENCIES =================
!pip install -q ddgs requests beautifulsoup4 transformers sentence-transformers faiss-cpu ipywidgets

# ================= IMPORTS =================
from ddgs import DDGS
import requests
from bs4 import BeautifulSoup
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM, pipeline
from sentence_transformers import SentenceTransformer
import faiss
import numpy as np
import ipywidgets as widgets
from IPython.display import display


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.7/43.7 kB 1.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.1/4.1 MB 23.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.8/23.8 MB 24.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 27.9 MB/s eta 0:00:00


In [ ]:
# ================= LLM SETUP =================
model_name = "google/flan-t5-base"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSeq2SeqLM.from_pretrained(model_name)
generator = pipeline("text-generation", model=model, tokenizer=tokenizer)
# ================= EMBEDDING =================
embedder = SentenceTransformer("all-MiniLM-L6-v2")

Loading weights:   0%|          | 0/282 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
The model 'T5ForConditionalGeneration' is not supported for text-generation. Supported models are ['PeftModelForCausalLM', 'AfmoeForCausalLM', 'ApertusForCausalLM', 'ArceeForCausalLM', 'AriaTextForCausalLM', 'BambaForCausalLM', 'BartForCausalLM', 'BertLMHeadModel', 'BertGenerationDecoder', 'BigBirdForCausalLM', 'BigBirdPegasusForCausalLM', 'BioGptForCausalLM', 'BitNetForCausalLM', 'BlenderbotForCausalLM', 'BlenderbotSmallForCausalLM', 'BloomForCausalLM', 'BltForCausalLM', 'CamembertForCausalLM', 'LlamaForCausalLM', 'CodeGenForCausalLM', 'CohereForCausalLM', 'Cohere2ForCausalLM', 'CpmAntForCausalLM', 'CTRLLMHeadModel', 'CwmForCausalLM', 'Data2VecTextForCausalLM', 'DbrxForCausalLM', 'DeepseekV2ForCausalLM', 'DeepseekV3ForCausalLM', 'DiffLl

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [ ]:
# ================= SEARCH AGENT =================
def search_agent(query, max_results=5):
    urls = []
    with DDGS() as ddgs:
        for r in ddgs.text(query, max_results=max_results):
            urls.append(r["href"])
    return urls

# ================= READER AGENT =================
def reader_agent(urls):
    articles = []
    for url in urls:
        try:
            r = requests.get(url, timeout=10)
            if "Please enable JS" in r.text:
                continue
            soup = BeautifulSoup(r.text, "html.parser")
            paragraphs = soup.find_all("p")
            text = " ".join([p.get_text() for p in paragraphs])
            if len(text) > 200:
                articles.append((url, text[:2000]))
        except:
            continue
    return articles


In [ ]:
# ================= BUILD VECTOR INDEX (RAG) =================
def build_index(articles):
    docs = [text for url, text in articles]
    embeddings = embedder.encode(docs)
    dimension = embeddings.shape[1]
    index = faiss.IndexFlatL2(dimension)
    index.add(np.array(embeddings))
    return index, docs

# ================= RETRIEVER =================
def retrieve(query, index, docs, k=3):
    query_embedding = embedder.encode([query])
    distances, indices = index.search(np.array(query_embedding), k)
    return [docs[i] for i in indices[0]]

In [ ]:
# ================= SUMMARIZER AGENT =================
def summarizer_agent(query, context):
    combined_context = "\n\n".join(context[:3])
    prompt = f"""
Answer the question clearly and briefly using the context below.

Question:
{query}

Context:
{combined_context}

Answer:
"""
    result = generator(prompt, max_new_tokens=120, do_sample=True, temperature=0.7)
    return result[0]["generated_text"].strip()

In [ ]:
# ================= RESULT DISPLAY =================
def show_results(answer, sources):
    print("\n===== FINAL ANSWER =====\n")
    print(answer)
    print("\n===== SOURCES =====\n")
    for s in sources:
        print(s)

# ================= ROUTER AGENT =================
def router_agent(query):
    query = query.lower()
    if "http://" in query or "https://" in query:
        return "url"
    elif "latest" in query or "news" in query or "current" in query:
        return "search"
    else:
        return "rag"


In [ ]:
def run_agent(query, max_results=5):
    decision = router_agent(query)
    print(f"\n🧠 Router decision: {decision}\n")

    if decision == "url":
        print("🌐 Processing URL...\n")
        urls = [query.strip()]
        articles = reader_agent(urls)

    elif decision == "search":
        print("🔎 Searching web...\n")
        urls = search_agent(query, max_results=max_results)
        articles = reader_agent(urls)

    else:
        print("📚 Using RAG knowledge...\n")
        urls = search_agent(query, max_results=max_results)
        articles = reader_agent(urls)

    if len(articles) == 0:
        print("No readable content found.")
        return

    print("📚 Building knowledge index...\n")
    index, docs = build_index(articles)

    print("🧠 Retrieving relevant context...\n")
    context = retrieve(query, index, docs)

    print("✍ Generating answer...\n")
    answer = summarizer_agent(query, context)

    print("\n===== FINAL ANSWER =====\n")
    print(answer)

    result_boxes = []

    for i, (url, text) in enumerate(articles, 1):

        box = widgets.Output(
            layout=widgets.Layout(
                border="2px solid #4CAF50",
                padding="10px",
                margin="10px"
            )
        )

        with box:
            print(f"📌 Source: {url}\n")
            print(text[:500], "...\n")

        result_boxes.append(box)

    display(widgets.VBox(result_boxes))

In [ ]:
# ================= USER INTERFACE =================
query_box = widgets.Text(
    value='',
    placeholder='Write your question...',
    description='Query:',
    layout=widgets.Layout(width='60%')
)

results_slider = widgets.IntSlider(
    value=5,
    min=1,
    max=10,
    step=1,
    description='Number of results:',
    style={'description_width': 'initial'},
    layout=widgets.Layout(width='40%')
)

search_button = widgets.Button(
    description="🔍 Search",
    button_style="info",
    layout=widgets.Layout(width='20%')
)

output_box = widgets.Output()

def on_submit(change):
    with output_box:
        output_box.clear_output()
        run_agent(change.value, results_slider.value)

def on_button_click(b):
    with output_box:
        output_box.clear_output()
        run_agent(query_box.value, results_slider.value)

query_box.on_submit(on_submit)
search_button.on_click(on_button_click)

display(widgets.VBox([
    widgets.HBox([query_box, search_button]),
    results_slider,
    output_box
]))
